<a href="https://colab.research.google.com/github/Sakthimurugavel/-Alpha-beta-pruning-of-Minimax-Search-Algorithm/blob/main/MonteCarloControlExp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import warnings ; warnings.filterwarnings('ignore')

import gym, gym_walk
import numpy as np

import random
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
np.set_printoptions(suppress=True)
random.seed(123); np.random.seed(123)

In [1]:
pip install git+https://github.com/mimoralea/gym-walk#egg=gym-walk

  Cloning https://github.com/mimoralea/gym-walk to /tmp/pip-install-610irt2r/gym-walk_2e285aad59cb4c1fa121797a871621eb
  Running command git clone --filter=blob:none --quiet https://github.com/mimoralea/gym-walk /tmp/pip-install-610irt2r/gym-walk_2e285aad59cb4c1fa121797a871621eb
  Resolved https://github.com/mimoralea/gym-walk to commit b915b94cf2ad16f8833a1ad92ea94e88159279f5
  Preparing metadata (setup.py) ... done
  Created wheel for gym-walk: filename=gym_walk-0.0.2-py3-none-any.whl size=5377 sha256=6419766be58167ab9ad55c3964ba626d7de2a54442b10ac7b89a07672c99d3dc
  Stored in directory: /tmp/pip-ephem-wheel-cache-acs5yvwj/wheels/bf/23/e5/a94be4a90dd18f7ce958c21f192276cb01ef0daaf2bc66583b
Successfully built gym-walk


In [2]:
def print_policy(pi, P, action_symbols=('<', 'v', '>', '^'), n_cols=4, title='Policy:'):
    print(title)
    arrs = {k:v for k,v in enumerate(action_symbols)}
    for s in range(len(P)):
        a = pi[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), arrs[a].rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [3]:
def print_state_value_function(V, P, n_cols=4, prec=3, title='State-value function:'):
    print(title)
    for s in range(len(P)):
        v = V[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), '{}'.format(np.round(v, prec)).rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [15]:
env = gym.make('FrozenLake-v1')
P = env.env.P
init_state = env.reset()
#goal_state = 6
#LEFT, RIGHT = range(2)

In [16]:
P

{0: {0: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  2: [(0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)],
  3: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 1: {0: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False)],
  2: [(0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  3: [(0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 2:

Exponentially decaying schedule


In [20]:
def decay_schedule(
    init_value, min_value, decay_ratio,
    max_steps, log_start=-2, log_base=10):

    values = np.zeros(max_steps)
    decay_steps = int(max_steps * decay_ratio)

    for step in range(max_steps):
        if step < decay_steps:
            # logarithmic decay from init_value to min_value
            progress = step / decay_steps
            values[step] = min_value + (init_value - min_value) * \
                (1 - np.log10(log_start + progress*(1 - log_start)) / np.log10(log_base))
        else:
            values[step] = min_value

    return values


Exploratory Policy Trajectories

In [ ]:
from itertools import count
def generate_trajectory(
    select_action, Q, epsilon,
    env, max_steps=200):
  done, trajectory = False, []

  #Write your code here

  return np.array(trajectory, np.object)

Monte Carlo control

In [22]:
def mc_control(env, gamma=1.0,
               init_alpha=0.5, min_alpha=0.01, alpha_decay_ratio=0.5,
               init_epsilon=1.0, min_epsilon=0.1, epsilon_decay_ratio=0.9,
               n_episodes=3000, max_steps=200, first_visit=True):

    nS, nA = env.observation_space.n, env.action_space.n

    # Initialize Q-values and policy
    Q = np.zeros((nS, nA))
    pi = np.zeros(nS, dtype=int)

    # Decay schedules for alpha and epsilon
    alpha_values = np.linspace(init_alpha, min_alpha, int(n_episodes * alpha_decay_ratio))
    alpha_values = np.pad(alpha_values, (0, n_episodes - len(alpha_values)), 'edge')

    epsilon_values = np.linspace(init_epsilon, min_epsilon, int(n_episodes * epsilon_decay_ratio))
    epsilon_values = np.pad(epsilon_values, (0, n_episodes - len(epsilon_values)), 'edge')

    # Loop over episodes
    for ep in range(n_episodes):
        state = env.reset()
        episode = []

        # Generate an episode
        for t in range(max_steps):
            epsilon = epsilon_values[ep]
            # Epsilon-greedy action
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(Q[state])

            next_state, reward, done, _ = env.step(action)
            episode.append((state, action, reward))
            state = next_state

            if done:
                break

        # Track returns
        G = 0
        visited_states = set()
        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = gamma * G + r

            if first_visit and (s, a) in visited_states:
                continue
            visited_states.add((s, a))

            alpha = alpha_values[ep]
            Q[s, a] += alpha * (G - Q[s, a])

        # Update policy
        for s in range(nS):
            pi[s] = np.argmax(Q[s])

    # Compute state-value function
    V = np.max(Q, axis=1)

    return Q, V, pi


In [24]:
optimal_Q, optimal_V, optimal_pi = mc_control (env)
print_state_value_function(optimal_Q, P, n_cols=4, prec=2, title='Action-value function:')
print_state_value_function(optimal_V, P, n_cols=4, prec=2, title='State-value function:')
print_policy(optimal_pi, P)

Action-value function:
| 00 [0.14 0.17 0.22 0.13] | 01 [0.09 0.12 0.09 0.23] | 02 [0.24 0.11 0.1  0.09] | 03 [0.01 0.01 0.   0.11] |
| 04 [0.25 0.12 0.13 0.09] |           | 06 [0.22 0.08 0.06 0.07] |           |
| 08 [0.15 0.12 0.06 0.25] | 09 [0.1  0.31 0.12 0.07] | 10 [0.42 0.06 0.18 0.03] |           |
|           | 13 [0.07 0.34 0.26 0.19] | 14 [0.34 0.68 0.57 0.38] |           |
State-value function:
| 00   0.22 | 01   0.23 | 02   0.24 | 03   0.11 |
| 04   0.25 |           | 06   0.22 |           |
| 08   0.25 | 09   0.31 | 10   0.42 |           |
|           | 13   0.34 | 14   0.68 |           |
Policy:
| 00      > | 01      ^ | 02      < | 03      ^ |
| 04      < |           | 06      < |           |
| 08      ^ | 09      v | 10      < |           |
|           | 13      v | 14      v |           |
